# 基于 LLM 的数据预处理：Netflix Titles

本方案使用 **Netflix Titles** 数据集，完成：
1. 数据质量问题识别；
2. 传统预处理方法；
3. LLM 缺失值填补；
4. LLM 文本结构化；
5. Prompt 策略对比（Zero-shot / Few-shot）。


In [1]:
import os
import json
import time
import numpy as np
import pandas as pd

DATA_PATH = '../data/netflix_titles.csv'
df = pd.read_csv(DATA_PATH)
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,81145628,Movie,Norm of the North: King Sized Adventure,"Richard Finn, Tim Maltby","Alan Marriott, Andrew Toth, Brian Dobson, Cole...","United States, India, South Korea, China","September 9, 2019",2019,TV-PG,90 min,"Children & Family Movies, Comedies",Before planning an awesome wedding for his gra...
1,80117401,Movie,Jandino: Whatever it Takes,NaN,Jandino Asporaat,United Kingdom,"September 9, 2016",2016,TV-MA,94 min,Stand-Up Comedy,Jandino Asporaat riffs on the challenges of ra...
2,70234439,TV Show,Transformers Prime,NaN,"Peter Cullen, Sumalee Montano, Frank Welker, J...",United States,"September 8, 2018",2013,TV-Y7-FV,1 Season,Kids' TV,"With the help of three human allies, the Autob..."
3,80058654,TV Show,Transformers: Robots in Disguise,NaN,"Will Friedle, Darren Criss, Constance Zimmer, ...",United States,"September 8, 2018",2016,TV-Y7,1 Season,Kids' TV,When a prison ship crash unleashes hundreds of...
4,80125979,Movie,#realityhigh,Fernando Lebrija,"Nesta Cooper, Kate Walsh, John Michael Higgins...",United States,"September 8, 2017",2017,TV-14,99 min,Comedies,When nerdy high schooler Dani finally attracts...


## 1. 数据质量问题分析

In [2]:
profile = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(df[c].dtype) for c in df.columns],
    'missing_count': [int(df[c].isna().sum()) for c in df.columns],
    'missing_ratio': [round(df[c].isna().mean(), 4) for c in df.columns],
    'nunique': [int(df[c].nunique(dropna=True)) for c in df.columns],
}).sort_values(['missing_count', 'column'], ascending=[False, True])
profile

,column,dtype,missing_count,missing_ratio,nunique
3,director,object,1969,0.3158,3301
4,cast,object,570,0.0914,5469
5,country,object,476,0.0764,554
6,date_added,object,11,0.0018,1524
8,rating,object,10,0.0016,14
11,description,object,0,0.0000,6226
9,duration,object,0,0.0000,201
10,listed_in,object,0,0.0000,461
7,release_year,int64,0,0.0000,72
0,show_id,int64,0,0.0000,6234


## 2. 传统预处理

In [3]:
# duration 解析
duration_parts = df['duration'].astype(str).str.extract(r'(?P<duration_value>\d+)\s+(?P<duration_unit>min|Season|Seasons)')
df['duration_value'] = pd.to_numeric(duration_parts['duration_value'], errors='coerce')
df['duration_unit'] = duration_parts['duration_unit']

# 日期标准化
df['date_added_parsed'] = pd.to_datetime(df['date_added'], errors='coerce')
df['year_added'] = df['date_added_parsed'].dt.year
df['month_added'] = df['date_added_parsed'].dt.month

df[['duration', 'duration_value', 'duration_unit', 'date_added', 'date_added_parsed']].head()

,duration,duration_value,duration_unit,date_added,date_added_parsed
0,90 min,90,min,"September 9, 2019",2019-09-09
1,94 min,94,min,"September 9, 2016",2016-09-09
2,1 Season,1,Season,"September 8, 2018",2018-09-08
3,1 Season,1,Season,"September 8, 2018",2018-09-08
4,99 min,99,min,"September 8, 2017",2017-09-08


## 3. 缺失值填补实验：传统方法 vs LLM

这里选择 `rating` 作为实验字段。原因：
- 缺失量不大，便于控制实验成本；
- 可以用“遮盖已知值”的方式做客观评估。


In [7]:
known = df[df['rating'].notna()].copy()
sample = known.sample(100, random_state=42).copy()
sample['gold_rating'] = sample['rating']
sample['rating'] = np.nan

global_mode = df['rating'].mode(dropna=True).iloc[0]
type_mode = df.groupby('type')['rating'].agg(lambda s: s.mode(dropna=True).iloc[0])

sample['pred_global_mode'] = global_mode
sample['pred_type_mode'] = sample['type'].map(type_mode).fillna(global_mode)

acc_global = (sample['gold_rating'] == sample['pred_global_mode']).mean()
acc_type = (sample['gold_rating'] == sample['pred_type_mode']).mean()

print('global mode accuracy =', round(acc_global, 4))
print('type mode accuracy   =', round(acc_type, 4))

global mode accuracy = 0.36
type mode accuracy   = 0.36


## 4. LLM Prompt 设计

### Zero-shot：rating 推断
```text
你是一个数据预处理助手。请根据以下 Netflix 节目的上下文信息，推断最可能的 age rating（如 TV-MA、TV-14、PG、PG-13、R、TV-PG 等）。

要求：
1. 只能输出一个 rating 标签；
2. 如果信息不足，也必须给出最可能结果；
3. 不要输出解释。

输入：
title: {title}
type: {type}
release_year: {release_year}
listed_in: {listed_in}
description: {description}
```

### Few-shot：rating 推断
```text
你是一个数据预处理助手，需要根据上下文推断 Netflix 节目的 age rating。

示例1：
title: Example Family Movie
type: Movie
release_year: 2018
listed_in: Children & Family Movies, Comedies
description: A heartwarming family comedy about friendship and school life.
输出：PG

示例2：
title: Example Crime Show
type: TV Show
release_year: 2020
listed_in: Crime TV Shows, TV Dramas
description: Detectives investigate brutal murders involving drugs and gang violence.
输出：TV-MA

现在请判断：
title: {title}
type: {type}
release_year: {release_year}
listed_in: {listed_in}
description: {description}
输出：
```


## 5. LLM 文本结构化

目标：把 `description` 变成结构化 JSON。


In [8]:
def keyword_structuring_baseline(description: str) -> dict:
    text = str(description).lower()
    risk_tags = []
    if any(k in text for k in ['violence', 'murder', 'crime', 'drug', 'war']):
        risk_tags.append('violence/crime')
    if any(k in text for k in ['love', 'romance', 'relationship']):
        main_theme = 'love/relationship'
    elif any(k in text for k in ['family', 'parents', 'children', 'school']):
        main_theme = 'family/growth'
    elif any(k in text for k in ['adventure', 'journey', 'quest', 'magic']):
        main_theme = 'adventure/fantasy'
    else:
        main_theme = 'other'

    if any(k in text for k in ['kids', 'child', 'school', 'family']):
        target_audience = 'family/teen'
    else:
        target_audience = 'general'

    if any(k in text for k in ['dark', 'murder', 'war', 'crime']):
        mood = 'dark'
    elif any(k in text for k in ['funny', 'comedy', 'hilarious']):
        mood = 'light'
    else:
        mood = 'neutral'

    return {
        'main_theme': main_theme,
        'target_audience': target_audience,
        'mood': mood,
        'risk_tags': risk_tags,
    }

sample_text = df[['title', 'description']].head(5).copy()
sample_text['baseline_json'] = sample_text['description'].apply(lambda x: json.dumps(keyword_structuring_baseline(x), ensure_ascii=False))
sample_text

,title,description,baseline_json
0,Norm of the North: King Sized Adventure,Before planning an awesome wedding for his gra...,"{""main_theme"": ""other"", ""target_audience"": ""ge..."
1,Jandino: Whatever it Takes,Jandino Asporaat riffs on the challenges of ra...,"{""main_theme"": ""other"", ""target_audience"": ""fa..."
2,Transformers Prime,"With the help of three human allies, the Autob...","{""main_theme"": ""other"", ""target_audience"": ""ge..."
3,Transformers: Robots in Disguise,When a prison ship crash unleashes hundreds of...,"{""main_theme"": ""other"", ""target_audience"": ""ge..."
4,#realityhigh,When nerdy high schooler Dani finally attracts...,"{""main_theme"": ""family/growth"", ""target_audien..."


### Zero-shot：description -> JSON
```text
请把下面的 Netflix 节目简介结构化为 JSON，并严格输出 JSON，不要输出其他文字。

字段要求：
- main_theme: 主要主题，字符串
- target_audience: 目标受众，字符串
- mood: 情绪基调，字符串
- risk_tags: 风险标签，字符串列表

文本：
{description}
```

### Few-shot：description -> JSON
```text
你是一个文本结构化助手。请把节目简介提取成 JSON。

示例输入：
A young girl and her dragon friend embark on a magical journey to save their village.

示例输出：
{
  "main_theme": "friendship and adventure",
  "target_audience": "teen and family",
  "mood": "inspiring",
  "risk_tags": []
}

现在请处理下面文本，并严格输出 JSON：
{description}
```
